# Chapter 1 · Notebook 1 of 3 — Introducing Token Factory

**AI Agent Course** · Nebius Token Factory × NVIDIA

Over the three notebooks of this chapter, we build **our first controlled agentic loop**: a model that can request an action, have our Python code execute it, and use the result. This notebook covers the first two steps of that journey: **choosing a model** and **calling it from Python**. Notebook 2 adds the tool loop and Notebook 3 checks whether the loop is fast enough and cheap enough to be usable.

**Prerequisites:** Python 3.10+, basic familiarity with REST APIs. No GPU, no prior LLM experience assumed.

**By the end of this notebook you can:** navigate the Token Factory catalog, read a model card and pick a model for a job, and make chat completion calls from Python — single-turn, styled with system prompts, and multi-turn.

## 1. What is Token Factory?

Open-sourced models are those in which all ingredients - including the weights, documentation, and often research - is publically available. Open source models are particularly powerful because they can be ran on your own servers.

In practice, that takes serious capacity and skills - GPU clusters, serving stacks, scaling, monitoring - and most teams have better things to do. **Managed inference** providers like Nebius Token Factory close that gap: they deploy and operate the open-source models for you. You can interact with these cheaper (and often equally as powerful!) models all in the playground or with simple API calls.

**In this course, we use Token Factory's public models which available to all users and billed per token which makes monitoring cost simple.**

## 2. Setup

1. Go to [tokenfactory.nebius.com](https://tokenfactory.nebius.com) and create an account.
2. Open **Get API Key → Create API key** and copy it (you can't view it again later).

<details>
<summary>Show screenshot: Token Factory home</summary>

![Token Factory home](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/token-factory-home.png)

</details>

<details>
<summary>Show screenshot: API Key creation</summary>

![API Key creation](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/api-key-creation.png)

</details>

Now we store that key somewhere our code can find it. We use a `.env` file: a plain text file that holds keys as `NAME=value` lines.
**Create it:**

1. In the same folder as this notebook, create a folder called `lesson` (if it doesn't exist yet).
2. Inside it, create a new plain text file named exactly `.env` (note: no name before the dot!). In VS Code or Jupyter: right-click the `lesson` folder, New File, type `.env`. From a terminal: `touch lesson/.env`.
3. Open it and paste in your key:

```
NEBIUS_API_KEY=paste-your-key-here
```

4. Save. That's it. No quotes, no spaces around the `=`.

**Why a `.env` file at all?** In real development, configuration that changes between people and machines (and *especially* secrets) lives outside your code. The `.env` file holds it; `load_dotenv()` reads it into environment variables. That way the same code runs on your laptop, your teammate's, and the server, and no key ever gets committed. **Key hygiene reminder:** keys live in `.env`, never in code, and `.env` goes in `.gitignore`. Never commit a notebook with a real key in it, unless you want someone stealing your Token Factory credits!

**Running in Google Colab?** Colab doesn't have your `.env`. Two options:

- Upload it: open the file browser (folder icon, left side) and drag your `.env` in, then point `load_dotenv()` at it with load_dotenv("path-to-your-env-file").
- Or use **Colab Secrets** (key icon, left side): add `NEBIUS_API_KEY` there, then run `from google.colab import userdata; os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")` instead of `load_dotenv()`.

One common gotcha: your operating system may hide files starting with a dot. If the file seems to vanish after you create it, it's still there. Your code will find it even if Finder or Explorer doesn't show it!

In [1]:
%pip install -q openai python-dotenv rich sympy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

from dotenv import load_dotenv
from rich import print

load_dotenv("lesson/.env")
%load_ext rich

# Colab users: comment the two lines above and use Colab Secrets instead:
# from google.colab import userdata
# os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")

assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in lesson/.env"
print("Keys loaded.")

Keys loaded.

## 3. The model catalog

The model catalog is the home for all models on Token Factory. Here, you can view basic information, and types of availability.

<details>
<summary>Show screenshot: Model catalog</summary>

![Model catalog](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-catalog.png)

</details>

The catalog hosts frontier open models: **GLM 5.2**, **Kimi 3** and the **NVIDIA Nemotron family**, among others.

## 4. The NVIDIA Nemotron ecosystem

Our flagship models for this course are the **NVIDIA Nemotron** series, a family of open-source models from NVIDIA's. The Nemotron series is NVIDIA's latest and most impressive models - spanning both small and efficient to large and frontier-performing.  The family shares training lineage and behavior, so you can prototype on a small one and scale up without rewriting anything. The three we care about:

| | **Nemotron 3 Nano** | **Nemotron 3 Super** | **Nemotron 3 Ultra** |
|---|---|---|---|
| Routing key | `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B` | `nvidia/nemotron-3-super-120b-a12b` | `nvidia/Nemotron-3-Ultra-550b-a55b` |
| Size | Smallest | Medium | Largest |
| Rough character | Fast and very cheap; great for everyday tasks, classification, drafting | The middle ground; strong general model for most workloads | The heavyweight; deepest reasoning, highest price |

A useful mental model: **start small, scale only when the task needs it.** The bigger the model, the more it costs per token and the slower it generates - and for a surprising share of real work, the small one is simply good enough!

**Our workhorse for this chapter is Nemotron 3 Nano.** We'll bring in its bigger siblings when we want a comparison, which will mimmic how you should scale with a real application.

## 5. Reading a model card

If you were deciding on an engine to be put in a car, you would likely read through different spec sheets until you found the one to fit your needs.

Every model in the catalog has a card - this is its spec sheet. The model cards are something you will come back to constantly: for the model string, for prices, for context limits, for what the model can and can't do. Need a fast car? You will pick the strongest - and likely most expensive - option. If you care about efficiency, you would choose one that is smaller and cheaper. The same choices will persist for the models you choose.

Click into **NVIDIA-Nemotron-3-Nano-30B-A3B**, scroll down, and click the model card.

<details>
<summary>Show screenshot: Nemotron 3 Nano model card</summary>

![Nemotron 3 Nano model card](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-card.png)

</details>

### Copying the model string

At the top of the endpoint properties you'll find the **routing key** - for our model it's `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B`, with a copy icon right next to it. **That exact string is what you pass as `model=` in every API call.** Always copy it from the card rather than typing it; a typo here is the single most common first error, and the API's "model not found" message won't tell you which character you got wrong.

### Reading the model card - what specs are important to us?

**Context window 262k.**
Find this number on the Nano card — it's the ceiling on everything the model can see at once: your prompt, the conversation history, any documents you paste in, *and* the output it generates. They all share this budget. Think of it as the model's working memory. A bigger window means you can hand it an entire codebase or a stack of contracts in one go; a smaller one means you'll have to chop things up and manage what the model gets to see.

**Pricing.**
Pricing is in terms of tokens, and there is a reason for the separation. Input tokens are cheap because the model ingests your whole prompt in a single go; output tokens cost more because the model generates them one at a time. Each one is a full trip through the model. Practical consequence: a chatty model that writes long answers costs you much more (think, what can we do to outputs to reduce this?). For scale: Ultra charges \$1.00 / \$3.00 for the same million tokens — roughly **15× more**.

**Modality: Text-to-text.**
What goes in and what comes out. Our model reads text and writes text, but scroll through the catalog and you'll see vision models (they accept images) and embedding models (they output vectors, not sentences). Checking this field takes two seconds and saves you from building half a project around a model that can't do what you assumed.

**Tool calling / Reasoning: Available.**
We'll return to what these mean later in the course — for now, just note that they're on the card, and that they'll matter a lot once we build agents.

**License.**
Open-weight does not mean do-whatever-you-want. Licenses differ on commercial use, redistribution, and what you can train on the outputs. Important to understand before putting a model in production, or starting work on something you may not be allowed to do.

### Exercise: pick a model by reading its card

Open two cards side by side: **`nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B`** and **`nvidia/Nemotron-3-Ultra-550b-a55b`**.

Two jobs. Tor each, decide which model you'd pick, and name the **one or two card fields** that settle it. One sentence of justification each ("it's bigger" doesn't count!):

1. **An everyday assistant** — rephrases emails, drafts short summaries, answers quick questions all day long.
2. **A contract analyst** — answers questions about 300-page legal documents pasted in whole, where a wrong answer is very costly.

*Hint: for one of these, the price gap is important. For the other, look at the context window and ask what a mistake costs.*

In [3]:
# Read the two cards from code: the catalog API returns the same fields the card page shows
from openai import OpenAI

BASE_URL = "https://api.tokenfactory.us-central1.nebius.com/v1/"
NANO = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B"
ULTRA = "nvidia/Nemotron-3-Ultra-550b-a55b"

catalog = OpenAI(base_url=BASE_URL, api_key=os.environ.get("NEBIUS_API_KEY"))
cards = {m.id: m.model_extra for m in catalog.models.list(extra_query={"verbose": "true"})}

for model_id in [NANO, ULTRA]:
    card = cards[model_id]
    price_in = float(card["pricing"]["prompt"]) * 1e6
    price_out = float(card["pricing"]["completion"]) * 1e6
    print(f"[bold]{model_id}[/bold]")
    print(f"  context window: {card['context_length']:,} tokens")
    print(f"  price per 1M:   ${price_in:.2f} input / ${price_out:.2f} output")
    print(f"  modality:       {card['architecture']['modality']} | features: {card['supported_features']}")

# Contract analyst: 300 pages at ~400 tokens/page (the rule of thumb from the video)
contract_tokens = 300 * 400
print(f"\n[bold]A 300-page contract is about {contract_tokens:,} tokens[/bold]")
for model_id in [NANO, ULTRA]:
    card = cards[model_id]
    share = contract_tokens / card["context_length"]
    cost = contract_tokens * float(card["pricing"]["prompt"])
    print(f"  {model_id}: fills {share:.0%} of the window, ${cost:.4f} input cost per question")

nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B

context window: 262,144 tokens

price per 1M:   $0.06 input / $0.24 output

modality:       text->text | features: ['tools', 'reasoning']

nvidia/Nemotron-3-Ultra-550b-a55b

...

price per 1M:   $1.00 input / $3.00 output

modality:       text->text | features: ['tools', 'reasoning']

A 300-page contract is about 120,000 tokens

nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B: fills 46% of the window, $0.0072 input cost per question

nvidia/Nemotron-3-Ultra-550b-a55b: fills 11% of the window, $0.1200 input cost per question

#### My answers

| Job | Pick | Card field(s) that settle it |
|---|---|---|
| **Everyday assistant** | **Nemotron 3 Nano** | **Pricing.** Nano costs \$0.06 / \$0.24 per 1M tokens versus Ultra's \$1.00 / \$3.00, so Ultra is about 17× more for input and 12.5× more for output. That's paid on every call, all day, for tasks Nano already handles well. The context window doesn't matter here: an email is a few hundred tokens. |
| **Contract analyst** | **Nemotron 3 Ultra** | **Context window, then pricing weighed against the cost of a mistake.** A 300-page contract is about 120k tokens. That's nearly half of Nano's 262k window before any conversation history or output, so follow-up questions and a second contract would overflow it. In Ultra's 1M window the same contract fills 11%. The pricing field shows Ultra's premium is about \$0.12 per question, which is trivial next to the cost of a wrong legal answer. |

## 6. First calls in the UI (Playground)

Before any code, use the **Token Factory playground**:

1. Open the model card for **Nemotron 3 Nano** and click **Go to playground**.
2. Send this prompt:

   > *A survey shows 62% of respondents prefer Product A and 38% prefer Product B, but the group that prefers A is three times smaller in the overall population. Which product actually wins overall? Show your reasoning briefly.*

3. Look at the **metrics below the response** - response time, token counts (it is OK to not fully understand these yet!).
<details>
<summary>Show screenshot: Nemotron response metrics</summary>

![Nemotron response metrics](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/request-metrics.png)

</details>

4. Now run the **exact same prompt** on **Nemotron 3 Ultra** side by side.

**What to compare:** the number of **output tokens** each model spent, how long each took, and the answers themselves - does the big model reason more carefully, or just write more? Did the small one get the right answer anyway? This is your first *evaluation*, and the honest answer to "which model is better" always starts with "for what purpose?".

#### Doing the comparison in code

The playground shows response time and token counts under each answer. The same numbers come back in every API response, so here's the Nano vs. Ultra comparison as a repeatable cell.

In [4]:
import time

SURVEY_PROMPT = (
    "A survey shows 62% of respondents prefer Product A and 38% prefer Product B, "
    "but the group that prefers A is three times smaller in the overall population. "
    "Which product actually wins overall? Show your reasoning briefly."
)

for model_id in [NANO, ULTRA]:
    start = time.perf_counter()
    resp = catalog.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": SURVEY_PROMPT}],
    )
    seconds = time.perf_counter() - start

    card = cards[model_id]
    usage = resp.usage
    cost = usage.prompt_tokens * float(card["pricing"]["prompt"]) + usage.completion_tokens * float(card["pricing"]["completion"])
    reasoning = (resp.choices[0].message.model_extra or {}).get("reasoning") or ""

    print(f"[bold]--- {model_id} ---[/bold]")
    print(f"time {seconds:.1f}s | output tokens {usage.completion_tokens:,} | reasoning {len(reasoning):,} chars | cost ${cost:.6f}")
    print(resp.choices[0].message.content.strip(), "\n")

--- nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B ---

time 3.0s | output tokens 812 | reasoning 2,234 chars | cost $0.000199

**Answer: Product B wins overall.**

**Reasoning (brief)**  

- Let \(p_A\) be the true fraction of the whole population that prefers A.  
- Let \(p_B\) be the true fraction that prefers B.  

The problem says *“the group that prefers A is three times smaller in the overall population.”*  
That means  

[
p_A = \frac{1}{3}\,p_B \quad\Longrightarrow\quad p_B = 3p_A .
\]

Since virtually everyone prefers either A or B, the two fractions must sum to 1:

[
p_A + p_B = 1 \;\Longrightarrow\; p_A + 3p_A = 1 \;\Longrightarrow\; 4p_A = 1 \;\Longrightarrow\; p_A = 0.25.
\]

Thus  

[
p_B = 3 \times 0.25 = 0.75.
\]

So 75 % of the overall population prefers Product B, while only 25 % prefers Product A. Therefore, **Product B wins
the overall preference**. The survey percentages (62 % vs. 38 %) reflect the respondents’ answers, but when you 
adjust for the actual population sizes, B’s share is substantially larger.

--- nvidia/Nemotron-3-Ultra-550b-a55b ---

time 2.2s | output tokens 633 | reasoning 0 chars | cost $0.001962

**Product B wins overall.**

**Reasoning:**
1.  **Define Population Groups:** Let the number of people who prefer Product A in the overall population be $x$.
2.  **Apply Constraint:** The problem states the group preferring A is "three times smaller" than the group 
preferring B. This means the group preferring B is $3x$.
3.  **Calculate Total Population:** Total population $= x + 3x = 4x$.
4.  **Determine True Preference:**
    *   Product A support: $x / 4x = \mathbf{25\%}$
    *   Product B support: $3x / 4x = \mathbf{75\%}$

The survey results (62%/38%) reflect a sampling bias (oversampling A-preferrers), not the actual population 
distribution. Product B wins with 75% overall support.

#### My notes on the comparison

The correct answer is **Product B**. If A-fans are a third the size of B-fans in the population, A has 25% of the market and B has 75%; the 62/38 survey split only reflects who answered.

From my run (one sample each, so treat the timings as rough):

| | Nano | Ultra |
|---|---|---|
| Correct? | Yes, B at 75% | Yes, B at 75%, and it names the sampling bias explicitly |
| Output tokens | 812 (about 2.2k characters of returned reasoning) | 633 |
| Time | 3.0 s | 2.2 s |
| Cost | \$0.00020 | \$0.00196 (about 10×) |

- **Both got it right.** On a puzzle this size, the small model is good enough, and Ultra's extra cost buys a slightly tighter write-up, not a better answer.
- **Nano wrote more, not less.** Most of its output tokens were reasoning, so for reasoning models, output tokens track how much the model thinks, not just how long the answer is.
- **Ultra returned no `reasoning` field**, yet it billed 633 output tokens for an answer of roughly 170 tokens. It probably reasons internally without returning that reasoning in the same field.
- **Speed depends on load and region.** Bigger isn't automatically slower on a single request; you'd need many runs to compare latency properly (that's Notebook 3's job).

## 7. The Chat Completions request

The Playground isn't for real applications, it's for *you* to evaluate a model. To use a model in real workflows, you need to call it from code, and the Token Factory API is exactly for that.

A Chat Completions request is the standard way to interact with models via API. It is a structured format for how we communicate to models and how they communicate back. A sample completions request for your model can be seen on the model card:

<details>
<summary>Show screenshot: Chat Completions example</summary>

![Chat Completions example](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/model-card-completions.png)

</details>

### Token Factory's API is OpenAI-compatible

If you ever went through any OpenAI API tutorial, this will be easy-peasy for you: everything works the same way, you just change the `base_url` and the model names. That compatibility is a big deal in practice:

- The entire ecosystem of tools works out of the box
- Drop-in migration from other providers
- Every tutorial on the internet works with a **one-line `base_url` change**

### 7.1 Anatomy of the request

Before the code, the pieces:

- **`model`** *(required)* — the routing key you copied from the card.
- **`messages`** *(required)* — the conversation as a list of roles. **Key note: the API is stateless — you send the whole conversation history every time.** The roles:
  - **`system`** — instructions for how the model should behave. Less about giving it expertise, more about **tone of voice and style**: the same question answered by "a formal analyst" and "a casual intern" reads completely differently. We'll see this live in a minute. Set once, at the top.
  - **`user`** — what the human says. Your prompts go here!
  - **`assistant`** — what the model said. You append its previous replies here so it remembers the conversation.
  - **`tool`** — results from function calls, which we'll meet in the next notebook.
- **Sampling params** *(optional)* — `temperature`, `top_p`, `max_tokens`. We will not worry about these for now.
- **The response object** — what comes back. We'll open one up right after the first call.

### 7.2 Minimal call

The model card provides a ready-made snippet ("Use model in code" - the one below is adapted from it), so you never have to write this boilerplate from memory:

In [5]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.us-central1.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "In two sentences, what does NVIDIA's Nemotron model family target?"

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

print(response.choices[0].message.content)

NVIDIA's Nemotron model family targets enterprise customers seeking to build custom AI applications by providing 
pre-trained, enterprise-grade foundation models. It enables businesses to fine-tune these models efficiently for 
specific industry workflows without extensive AI development from scratch.

### 7.3 What came back? A one-time tour of the response

The response isn't just text - it's a structured object, and a few of its fields matter for everything we do later. Let's dump the whole thing **once**, look around, and then never print it in full again:

In [6]:
# One-time tour — after this cell, we only ever print .content
print(response.to_json())

# The fields worth remembering:
print("---")
print("content:       ", response.choices[0].message.content[:60], "...")   # the answer
print("finish_reason: ", response.choices[0].finish_reason)                  # why it stopped ("stop" = natural end, "length" = hit max_tokens)
print("usage:         ", response.usage)                                     # tokens in/out — this becomes our COST signal in Notebook 3

{
  "id": "chatcmpl-96b9bb90-0ef8-4cbe-9dae-bbafa3c8d736",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "\nNVIDIA's Nemotron model family targets enterprise customers seeking to build custom AI 
applications by providing pre-trained, enterprise-grade foundation models. It enables businesses to fine-tune these
models efficiently for specific industry workflows without extensive AI development from scratch.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [],
        "reasoning": "Hmm, the user is asking about NVIDIA's Nemotron model family in just two sentences. They 
specifically want to know what it targets, and they've emphasized conciseness since they're part of a 
market-research team. \n\nI recall that Nemotron is NVIDIA's generative AI model series designed for enterprises to
create custom AI applications. The key target is enterprise customers who need to fine-tune models for specific 
workflows without building from scratch. \n\nThe user's strict two-sentence limit means I must be razor-focused. 
First sentence should cover the core purpose (enterprise custom AI), second should highlight the key benefit 
(pre-trained models reducing development effort). \n\nI should avoid technical jargon since it's for market 
research - they'll care about customer segments, not model architecture. Also must resist adding extra details like
pricing or competitors since they want absolute brevity. \n\nDouble-checking: yes, NVIDIA's official docs position 
Nemotron as the \"engine\" for enterprise AI customization. No fluff needed. \n\n*Crafting response mentally*: \n1.
\"NVIDIA's Nemotron targets enterprise customers seeking to build custom AI models...\" \n2. \"...using pre-trained
models to accelerate development in specific business workflows.\" \n\nPerfect - exactly two sentences, covers who 
and why, no extra words.\n"
      },
      "stop_reason": null,
      "token_ids": null,
      "routed_experts": null
    }
  ],
  "created": 1792142456,
  "model": "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
  "object": "chat.completion",
  "moderation": null,
  "service_tier": null,
  "system_fingerprint": "vllm-0.22.0-b08c96cd",
  "usage": {
    "completion_tokens": 334,
    "prompt_tokens": 44,
    "total_tokens": 378,
    "completion_tokens_details": null,
    "prompt_tokens_details": null
  },
  "prompt_logprobs": null,
  "prompt_token_ids": null,
  "prompt_text": null,
  "kv_transfer_params": null
}

---

content:        
NVIDIA's Nemotron model family targets enterprise customers ...

finish_reason:  stop

usage:         
CompletionUsage(
    completion_tokens=334,
    prompt_tokens=44,
    total_tokens=378,
    completion_tokens_details=None,
    prompt_tokens_details=None
)

Three things to file away: **`content`** is the answer, **`finish_reason`** tells you *why* generation stopped (watch for `"length"` - it means the answer was cut off), and **`usage`** counts prompt and completion tokens - the number your bill is computed from. From here on, our cells print only the content, for readability.

### 7.4 System prompts: same question, two voices

The claim above was that system prompts are mostly about tone and style. Prove it! Same user question, two different personas:

In [7]:
QUESTION = "Is the specialty coffee market growing?"

for persona in [
    "You are a formal senior market analyst. Precise, measured, no exclamation points.",
    "You are an enthusiastic intern on the market-research team. Casual, punchy, emoji welcome.",
]:
    resp = client.chat.completions.create(
        model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
        messages=[
            {"role": "system", "content": persona},
            {"role": "user", "content": QUESTION},
        ],
    )
    print(f"[bold]--- {persona[:40]}... ---[/bold]")
    print(resp.choices[0].message.content, "\n")

--- You are a formal senior market analyst. ... ---

MarketGrowthYes
TrendPositive
DataRecent
CAGR7.8%
Revenue45B
ConsumerShift
UrbanDemand
PremiumWillingness
ThirdWaveExpansion
StoreGrowth
AcquisitionActivity
VCInvestment
ExportGrowth
PriceIndexUp
SupplyStable
CompetitionIntense
BarriersHigh
OpportunitiesInnovation
ProductDiversification
SustainabilityFocus
EthicalSourcing
RegionalExpansion
PremiumPricing
DirectTrade
CoffeeCertification
SpecialtyRetail
OnlineSales
SubscriptionModels
ConsumerEducation
DemographicShift
MillennialPreference
GenZSpending
HealthTrends
LatteArtPopularity
BrewingTech
ColdBrewGrowth
ReadyToDrink
PackagedCoffee
NicheFlavors
LimitedEdition
CoffeeTourism
DestinationCafes
CafeCulture
UrbanizationImpact
MillennialIncome
DisposableSpending
BrandLoyalty
CustomerRetention
DigitalMarketing
SocialMediaInfluence
InfluencerCollabs
SubscriptionServices
HomeBrewing
ApplianceSales
CoffeeSubscriptions
LoyaltyPrograms
PricingStrategy
MarginImprovement
CostControl
RetailLocation
FootTraffic
RealEstateTrends
MarketFragmentation
CompetitiveLandscape
StartupActivity
IncubatorSupport
FundingRounds
VentureCapital
PrivateEquity
IPOActivity
MarketSegments
ReadyToDrinkSegment
SingleOrigin
LightRoast
DarkRoast
QualityPerception
ConsumerPreferencesShift
FlavorProfilesDiversify
GrowthOutlookPositive
MarketOutlookPositive

--- You are an enthusiastic intern on the ma... ---

🚀 Absolutely! The specialty coffee market is on a serious upward trend—think double‑digit growth year‑over‑year, 
more boutique roasters popping up, and consumers craving premium experiences. ☕️💡  

- 2023 sales around **$45 B** globally, with a **10‑12%** CAGR projected through 2028.  
- Millennials & Gen Z driving demand for ethically sourced, unique flavor profiles.  
- Coffee‑shop density up **15%** in major cities—proof people are willing to pay for that perfect cup.  

Bottom line: the industry isn’t just growing, it’s exploding! 🌱📈 Keep that curiosity brewing!

Same facts, completely different voice. In an agent, the system prompt is where you pin down *how* it should behave - style, format, boundaries, etc.

### 7.5 Multi-turn: the API is stateless, so *you* carry the memory

Ask a follow-up question in a fresh request and the model has no idea what you're talking about - nothing is stored server-side. "Memory" is literally you resending the conversation. First, fully by hand, so you see every move:

In [8]:
# Turn 1: ask
history = [
    {"role": "system", "content": "You are a concise assistant for a market-research team."},
    {"role": "user", "content": "Name three fast-growing beverage categories."},
]
resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
answer_1 = resp.choices[0].message.content
print(answer_1)

# Turn 2: append the model's answer, then our follow-up, then resend EVERYTHING
history.append({"role": "assistant", "content": answer_1})
history.append({"role": "user", "content": "Which of those is cheapest to enter as a small brand?"})

resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
print("---")
print(resp.choices[0].message.content)   # it knows "those" only because we resent the history

**Fast‑growing beverage categories (2023‑2024):**

1. **Non‑alcoholic ready‑to‑drink (RTD) teas & infused waters** – low‑calorie, functional flavors and 
health‑focused branding are driving double‑digit growth worldwide.  
2. **Plant‑based milks & alternatives (e.g., oat, almond, pea)** – expanding beyond coffee shops into grocery 
aisles; Oat Milk alone posted ~30 % YoY sales growth in the U.S.  
3. **Hard seltzers & low‑ABV ready‑to‑drink cocktails** – driven by younger consumers seeking lighter, 
“sessionable” options; the category grew > 40 % globally in the past year.  

These segments are attracting strong investment and retail shelf space, making them prime targets for new product 
development and brand extensions.

---

**Best bet for a low‑cost start‑up: Non‑alcoholic Ready‑to‑Drink (RTD) teas & infused waters**

| Factor | Why RTD teas/ infused waters win on cost |
|--------|------------------------------------------|
| **Ingredient simplicity** | A handful of tea leaves (or flavor extracts), water, natural sweetener (e.g., stevia,
honey), and a stabilizer/fizz (optional). No dairy, grains, or fermentation bases required. |
| **Production facilities** | Small‑batch bottling can be done in a co‑packer or even a shared‑use kitchen with 
minimal line modifications. You don’t need a mash‑tun, brewhouse, or pasteurization equipment for dairy‑heavy 
milks. |
| **Regulatory path** | Most markets classify these as “non‑alcoholic, non‑preserved beverages.” Labeling and 
safety testing (e.g., water activity, pH) are straightforward, avoiding the extra alcohol licensing that hard 
seltzers face. |
| **Packaging** | PET or recyclable aluminum cans are cheap to source in small volumes (500 ml‑330 ml). Simple 
label designs can be printed digitally; no need for heavy‑weight glass bottles or special caps. |
| **Distribution footprint** | You can start with “direct‑to‑consumer” (online DTC) and small‑scale retail (local 
health‑food stores, farmer’s markets). Many retailers will stock low‑margin RTD drinks without demanding slotting 
fees or large order volumes. |
| **Marketing & brand building** | Health‑focused storytelling (hydration, antioxidants, botanicals) resonates with
younger, wellness‑savvy consumers and requires modest spend on influencer or social media campaigns. |
| **Capital requirements** | A modest prototype budget (USD 5k‑10k) can cover initial flavor development, packaging
samples, and a small batch run (≈5,000–10,000 units). This is an order of magnitude lower than scaling a 
plant‑based milk line (which needs refrigeration and often a co‑packer with larger minimums) or a hard‑seltzer 
(requires alcohol licensing, higher‑volume fermentation, and stricter distribution channels). |

### Quick “starter‑kit” checklist

1. **Recipe & formulation** – 1–2 tea bases (e.g., hibiscus‑mint, lemon‑ginger) + natural sweetener + optional 
carbonation. Run a small test batch in a home‑brew kettle or partner kitchen.
2. **Co‑packer or shared‑use facility** – Look for contract bottlers specializing in non‑carbonated RTDs; many 
accept runs as low as 5,000 cans.
3. **Label & compliance** – Ensure water activity < 0.60 (or use a preservative) and label per local beverage 
regulations (ingredients, nutrition facts, allergen statements).
4. **Packaging sourcing** – Bulk PET cans (12‑oz) with simple shrink‑wrap or printed glass‑like labels; order 
12‑pack cartons for retail shelf.
5. **Pilot sales** – Launch on a DTC platform (Shopify, Etsy) + local farmer’s markets or health‑food co‑ops. Track
cost‑per‑unit and consumer feedback.
6. **Iterate & scale** – Use early sales data to refine flavor combos and negotiate larger co‑packer volumes (e.g.,
20‑50 k units) for a price break.

### Why the other two are more expensive

| Category | Minimum cost drivers |
|----------|----------------------|
| **Plant‑based milks** | Require refrigerated processing, emulsifiers, longer shelf‑life testing, and often a 
larger co‑packer minimum (≥ 20‑30 k units). Many retailers demand slotting fees and prefer established brands. |
| **Hard seltzers** | Must obtain an alcohol manufacturing license, manage fermentation (often 3‑6 weeks), run 
rigorous alcohol‑content testing, and meet stricter distribution agreements. Production equipment (fermenters, 
carbonation tanks) and higher volume minimums raise upfront capital. |

**Bottom line:** If cash flow, regulatory simplicity, and small‑batch flexibility are your top priorities, 
launching a low‑cost RTD tea or infused‑water line gives you the cheapest and fastest route to market as a solo or 
micro‑brand.

That works, but the appending pattern is always the same, so wrap it in a tiny loop and you have a chat:

In [9]:
history = [{"role": "system", "content": "You are a concise assistant for a market-research team."}]

def ask(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    resp = client.chat.completions.create(model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B", messages=history)
    answer = resp.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    return answer

print(ask("Name three fast-growing beverage categories."))
print("---")
print(ask("Which of those is cheapest to enter as a small brand?"))

**1. Ready‑to‑Drink (RTD) Flavored Waters & Functional Hydration**  
- **Why it’s booming:** Health‑conscious consumers are swapping sugary sodas for low‑calorie, vitamin‑fortified, or
adaptogenic waters.  
- **Growth cue:** CAGR ≈ 12‑15% (2022‑2027) in the U.S. and ≈ 10% globally, driven by millennials and Gen Z seeking
“hydration plus” benefits.

**2. Plant‑Based & Alt‑Milk Beverages (e.g., oat, almond, pea, oat‑based “milks” and “coffees”)**  
- **Why it’s booming:** Sustainability and dairy‑free diets are moving mainstream; new flavor extensions and 
barista‑grade formulations are expanding use beyond cereal.  
- **Growth cue:** CAGR ≈ 14‑18% (2021‑2026) for the U.S. market, with Asia‑Pacific showing the fastest adoption 
rates.

**3. Low‑/No‑Alcohol Ready‑to‑Serve (RTS) Cocktails & Hard Seltzers**  
- **Why it’s booming:** “Better‑for‑you” social drinking—lower ABV, fewer calories, and added botanicals appeal to 
younger drinkers who still want a cocktail experience.  
- **Growth cue:** CAGR ≈ 20‑25% (2022‑2027) in the U.S.; hard seltzer alone grew > 30% YoY in 2023, and low‑ABV RTD
cocktails are entering the 1‑2% ABV segment at double‑digit rates.  

These categories are expanding faster than the overall beverage market because they combine health trends, 
sustainability concerns, and premium‑flavored experiences that attract both new and existing consumers.

---

**The cheapest of the three to launch as a small‑brand is the RTD Flavored‑Water / Functional Hydration category.**

Below is a quick comparison that shows why it’s the lowest‑cost entry point, followed by the key cost‑drivers 
you’ll need to watch.

---

## 1. RTD Flavored‑Water / Functional Hydration  
| Factor | Why it’s low‑cost | Typical budget for a 1‑year launch |
|--------|-------------------|-----------------------------------|
| **Ingredient cost** | Water + food‑grade flavor concentrates + modest functional boosts (e.g., electrolytes, 
vitamins, botanicals). These are commodity‑grade and can be sourced in 50‑lb bulk drums for pennies per litre. | 
**$30‑$60 k** for raw material inventory (enough for ~150 k L of ready‑to‑sell product). |
| **Formulation & R&D** | Simple pH‑stable recipes; no emulsifiers, no high‑pressure processing, no fermentation. 
Small‑batch kitchen‑scale testing can be done on a standard cooktop. | **$5‑$15 k** for a contract formulator or a 
freelance food‑tech consultant. |
| **Regulatory path** | Classified as a non‑ alcoholic beverage; usually falls under “food” rather than “alcoholic 
beverage” regulations (no TTB approval, no excise tax). Only a standard FDA food‑facility registration and labeling
compliance are needed. | **$2‑$5 k** for filing, label compliance consulting, and basic safety testing (e.g., 
microbiology, heavy‑metal screening). |
| **Packaging** | 12‑oz or 16‑oz PET bottles or recyclable pouches; can start with a private‑label bottle‑filling 
line or a “filler‑as‑a‑service” (FAAS) vendor. No glass, cans, or aluminum required for the first run. | **$10‑$25 
k** for a 5,000‑piece starter run (incl. bottle, cap, label, and simple capping equipment). |
| **Manufacturing** | FAAS or co‑packer can fill 2,000‑10,000 L per batch with minimal setup fees; you pay per 
litre, no expensive capital equipment. | **$0.20‑$0.35 / L** (including labor, filling, QC). |
| **Distribution** | Direct‑to‑consumer (DTC) via ecommerce, small local grocery or health‑store placements, or 
subscription boxes. No heavy‑weight distribution network needed initially. | **$5‑$12 k** for initial market‑entry 
(samples, POP materials, modest sales‑rep outreach). |
| **Marketing** | Influencer & social‑media push, sampling at gyms/colleges, low‑budget POS displays. | **$5‑$10 
k** for creative content, small media spend, and event sponsorship. |
| **Total first‑year estimate** | **≈ $55‑$130 k** (including a modest safety‑stock buffer). |  |

*Bottom‑line:* You can launch with a **few thousand dollars** in inventory, keep overhead low, and avoid the 
biggest regulatory and capital hurdles that the other two categories impose.

---

## 2. Plant‑Based / Alt‑Milk Beverages  
| Factor | Cost impact |
|--------|-------------|
| **Ingredients** | Nuts, oats, peas, or legumes are bulkier and more expensive per kg than flavor concentrates; 
you also need emulsifiers, stabilizers, and often a “cream‑like” texture agent. |
| **Formulation & R&D** | Requires lab‑scale trials to achieve stable emulsions, appropriate viscosity, and taste 
balance. May need a food‑science consultant (higher hourly rates). |
| **Regulatory** | Still a “food” product, but you need to meet “milk‑analogue” labeling rules, nutritional panels,
and sometimes a “contains nuts” allergen statement. |
| **Packaging** | Typically requires **aseptic cartons (e.g., 6‑oz or 1‑L cartons)**, which are pricier than PET 
bottles and often have minimum order quantities. |
| **Manufacturing** | Needs at least a **small‑scale homogenizer & pasteurizer**; many co‑packers charge set‑up 
fees and higher per‑litre rates because of equipment wear‑and‑tear. |
| **Distribution** | Shelf‑stable cartons need placement in grocery aisles; that usually means higher trade‑spend 
and slotting fees. |
| **Total first‑year estimate** | **$150‑$350 k** (often higher if you target “barista‑grade” or fortified 
variants). |

*Why it’s costlier:*  Ingredient complexity, higher per

Notice what the loop implies about cost: every turn resends the *whole* history, so long conversations re-pay for everything already said.

### Exercise: try to create a request that does these tasks!

All of these use our market-research assistant. Fill in the `## What goes here?` gaps.

#### 1. Only produce 10 tokens or less per response

In [10]:
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Give me a one-line headline on the energy-drink market."

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
    max_tokens=10,  # hard cap on generated tokens
    # Nano is a reasoning model: with thinking on, its hidden reasoning spends the 10 tokens and content comes back None
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print(response.choices[0].message.content)
print("---")
print("finish_reason:", response.choices[0].finish_reason)  # what do you expect to see here?

Energy‑Drink Market Booms: $

---

finish_reason: length

**Answer:** `max_tokens=10`, and `finish_reason` is `"length"` because the headline gets cut off mid-sentence. A prompt like "answer in 10 tokens" isn't enough on its own; only `max_tokens` enforces the cap.

**Gotcha I hit:** with thinking left on, the 10 tokens all went to Nano's hidden reasoning (`message.reasoning`), and `content` came back as `None`. Turning thinking off with `chat_template_kwargs` gives the visible answer the whole budget. Reasoning tokens count toward `max_tokens` and toward `usage.completion_tokens`, so you pay for them too.

#### 2. Change the *voice*: make the assistant answer like a skeptical veteran analyst who distrusts hype

In [11]:
SYSTEM_PROMPT = (
    "You are a skeptical veteran market analyst with 30 years in the industry. "
    "You've watched countless 'booming' markets fizzle, so you distrust hype, press releases, and vendor-funded reports. "
    "Question the source and methodology behind every growth claim, separate real demand from marketing noise, "
    "and call out what the bulls are ignoring. Dry, blunt, and unimpressed. No exclamation points."
)
USER_MESSAGE = "Is the specialty coffee market growing?"

response = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

print(response.choices[0].message.content)

The specialty coffee market is growing at an estimated 5-7% annual rate globally, per industry reports from firms 
like Statista and IBISWorld. But growth is overstated in public discourse. The methodology behind these figures is 
often self-reported by trade associations or vendors with vested interests—SPECIALTY COFFEE ASSOCIATION data isn't 
independently verified, and many "specialty" businesses misuse the term for basic retail coffee. Real demand growth
is concentrated in high-density urban markets, not broadly across demographics. The actual shift in consumer 
behavior—willingness to pay premium prices—is narrow and fragile. Most "growth" comes from opening new cafes, not 
sustained profitability. Over 60% of new specialty cafes fail within three years, according to internal industry 
surveys. The bulls ignore that premium pricing is a luxury, not a scalable model. Price inflation in the sector is 
driven by operational costs, not consumer demand. The market is saturating rapidly in key regions like the U.S. and
Europe, with new entrants competing on price, not quality. What the growth narrative misses is that most consumers 
don't pay $5+ for a cup; they're just exploring a trend. The real story isn't growth—it's the fragile economics of 
a niche segment that can't scale without collapsing margins. The data is noisy, the drivers are weak, and the hype 
is outpacing reality.

#### 3. Challenge! Make the model produce 10 identical responses with:

```python
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Produce 5 bullet points on why cold brew is winning the coffee market."
```

**Hint: you may need to use a loop and a certain parameter that makes the model more deterministic (less creative)**

In [12]:
SYSTEM_PROMPT = "You are a concise assistant for a market-research team."
USER_MESSAGE = "Produce 5 bullet points on why cold brew is winning the coffee market."

responses = []
for _ in range(10):
    response = client.chat.completions.create(
        model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_MESSAGE},
        ],
        temperature=0,  # greedy decoding: always take the most likely next token
        seed=42,        # fixed seed in case anything is still sampled
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},  # no hidden reasoning to drift
    )
    responses.append(response.choices[0].message.content)

print(response.choices[0].message.content)
print("---")
print(f"unique responses: {len(set(responses))} / {len(responses)}")

- **Smooth, low‑acid profile** appeals to health‑conscious and sensitive‑stomach drinkers.  
- **Convenient ready‑to‑drink formats** dominate grocery shelves and boost on‑the‑go sales.  
- **Premium perception** allows higher price points and drives brand‑building opportunities.  
- **Versatile serving options** (milk, sweeteners, flavor infusions) attract diverse consumer tastes.  
- **Sustainability narrative** (longer shelf life, reduced waste) resonates with eco‑aware shoppers.

---

unique responses: 10 / 10

**What I found:** the intended answer is a loop plus `temperature=0` (with a fixed `seed`). In testing on Token Factory, though, the 10 responses still weren't byte-identical. The wording and bullet order drift between calls even with `temperature=0`, `seed`, `top_k=1` and thinking off, on both the regional and the global endpoint. Temperature itself is honored: at `temperature=2.0` the output turns into multilingual gibberish.

The likely cause is server-side non-determinism. Nano is a mixture-of-experts model served in shared batches, so tiny floating-point differences change which expert or token wins early, and the text diverges from there. **Takeaway:** `temperature=0` makes a hosted model *more* consistent, not identical. If you need the exact same output every time, cache the response; don't count on regenerating it.

## Wrap-up!

You created an account and an API key, and stored it in a dot-env file so it stays out of your code. You learned to read a model card - the routing key, the context window and the three things sharing it, the two pricing rates and why output costs more, modality, and the license. You tested a model in the playground and compared a small one against a large one on the same prompt. And then you built the same interaction in code: a single request, the response object with its content, finish reason, and usage, a system prompt to control voice, and a multi-turn conversation where you carry the history yourself.

Together, that means you can choose a model for a job, call it from Python, and hold a real conversation with it while knowing exactly what each request cost you.

You now have a working foundation - a model you've chosen deliberately, and the code to talk to it.
